In [ ]:
import ipywidgets.widgets as widgets
from IPython.display import display
import time
import numpy as np
from DOGZILLALib import DOGZILLA

# 初始化机器人
g_dog = DOGZILLA()
g_ENABLE_CHINESE = False

#  widgets名称配置
Name_widgets = {
    'Force_Balance_Mode': ("Force Feedback Balance", "力反馈找平模式"),
    'ON': ("ON", "已开启"),
    'OFF': ("OFF", "已关闭"),
    'Force_Calibrate': ("Calibrate Force", "校准受力零点")
}

# ---------------------- 力反馈核心参数配置 ----------------------
# 力控模式开关（全局变量）
g_force_balance_enabled = False
# 受力校准零点（存储各传感器的初始零点，需校准）
g_force_zero = [0.0] * 4  # 假设4个脚底力传感器（前左/前右/后左/后右）
# 目标受力范围（单位：N，根据机器人重量设定，如总重10kg，每个脚约25N）
g_target_force = 25.0
# 力控比例系数（P增益，调整响应速度，需调试）
g_force_kp = 0.8
# 最大关节力矩限制（防止过载，单位：N·m）
g_max_torque = 5.0
# 受力平衡阈值（小于该值视为已平衡，单位：N）
g_force_threshold = 2.0

# ---------------------- 控件创建 ----------------------
# 力反馈模式开关按钮
button_ForceBalance = widgets.Button(
    value=False,
    description=Name_widgets['Force_Balance_Mode'][g_ENABLE_CHINESE],
    button_style='success',
    tooltip='Click to enable/disable force feedback balance',
    icon='uncheck'
)

# 受力校准按钮（必须先校准零点，否则力控不准）
button_Calibrate = widgets.Button(
    value=False,
    description=Name_widgets['Force_Calibrate'][g_ENABLE_CHINESE],
    button_style='info',
    tooltip='Click to calibrate force sensor zero',
    icon='sync'
)

# 输出日志控件
output = widgets.Output()

# ---------------------- 核心功能函数 ----------------------
def calibrate_force_zero(b):
    """校准力传感器零点（每次启动或更换地面后需执行）"""
    global g_force_zero
    with output:
        print("\n=== 开始力传感器零点校准 ===")
        print("请确保机器人脚底未受力（悬空或轻触地面）...")
        time.sleep(1)
        
        # 采集10次传感器数据取平均值作为零点（根据实际传感器数量调整）
        force_samples = []
        for i in range(10):
            # 假设g_dog.get_foot_force()返回4个脚底力传感器原始值（需根据硬件API修改）
            raw_force = g_dog.get_foot_force()  # 示例：返回 [f1, f2, f3, f4]
            force_samples.append(raw_force)
            time.sleep(0.05)
        
        # 计算零点（平均值）
        g_force_zero = np.mean(force_samples, axis=0).tolist()
        print(f"校准完成！零点值：{g_force_zero}")
        print("=== 校准结束 ===")

def force_balance_control():
    """力反馈找平核心控制逻辑（循环执行）"""
    global g_force_balance_enabled
    while g_force_balance_enabled:
        try:
            # 1. 读取力传感器数据（减去零点校准值）
            raw_force = g_dog.get_foot_force()  # 原始数据
            actual_force = [f - z for f, z in zip(raw_force, g_force_zero)]  # 实际受力（去零点）
            
            # 2. 计算每个脚的受力偏差（目标受力 - 实际受力）
            force_errors = [g_target_force - f for f in actual_force]
            
            # 3. 过滤微小偏差（避免高频抖动）
            force_errors = [e if abs(e) > g_force_threshold else 0.0 for e in force_errors]
            
            # 4. 计算关节调整力矩（P控制：力矩 = 偏差 × 比例系数）
            # 假设4个脚对应4组关节力矩（需根据机器人关节配置映射，这里是简化示例）
            joint_torques = [min(max(e * g_force_kp, -g_max_torque), g_max_torque) for e in force_errors]
            
            # 5. 发送力矩指令给机器人（关键：替换原IMU角度控制，改为力矩控制）
            # 注意：需根据机器人关节数量和配置调整力矩映射（这里是简化版）
            # 示例：前左/前右/后左/后右关节分别施加对应力矩
            g_dog.set_joint_torque(
                front_left=joint_torques[0],
                front_right=joint_torques[1],
                rear_left=joint_torques[2],
                rear_right=joint_torques[3]
            )
            
            # 6. 日志输出（可选，调试用）
            with output:
                print(f"力反馈控制 - 实际受力：{[round(f,1) for f in actual_force]} N | "
                      f"调整力矩：{[round(t,2) for t in joint_torques]} N·m")
        
        except Exception as e:
            with output:
                print(f"力控异常：{str(e)}")
            break
        
        # 控制频率（100Hz，根据硬件响应速度调整）
        time.sleep(0.01)

def on_force_balance_clicked(b):
    """力反馈模式开关回调"""
    global g_force_balance_enabled
    with output:
        if b.icon == 'uncheck':
            # 开启力反馈模式
            if all(z == 0.0 for z in g_force_zero):
                print("警告：未校准力传感器零点！请先点击校准按钮！")
                return
            
            g_force_balance_enabled = True
            b.icon = 'check'
            b.button_style = 'danger'
            print(f"=== {Name_widgets['Force_Balance_Mode'][g_ENABLE_CHINESE]} {Name_widgets['ON'][g_ENABLE_CHINESE]} ===")
            # 启动力控循环（注意：这里用线程避免阻塞UI）
            import threading
            threading.Thread(target=force_balance_control, daemon=True).start()
        else:
            # 关闭力反馈模式
            g_force_balance_enabled = False
            b.icon = 'uncheck'
            b.button_style = 'success'
            print(f"=== {Name_widgets['Force_Balance_Mode'][g_ENABLE_CHINESE]} {Name_widgets['OFF'][g_ENABLE_CHINESE]} ===")
            # 停止力控：关节力矩归零（或恢复默认姿态）
            g_dog.set_joint_torque(front_left=0, front_right=0, rear_left=0, rear_right=0)
            g_dog.imu(0)  # 关闭原IMU映射模式

# ---------------------- 绑定控件事件 ----------------------
button_ForceBalance.on_click(on_force_balance_clicked)
button_Calibrate.on_click(calibrate_force_zero)

# ---------------------- 显示控件 ----------------------
box_display = widgets.VBox([button_Calibrate, button_ForceBalance, output])
display(box_display)